In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def load_data():

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    return X, y, X_test


def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


def train_kfold_ensemble(X, y, X_test):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    test_predictions = np.zeros(len(X_test))
    rmse_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"\nTraining fold {fold+1}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=1.0,
            l1_ratio=0.7,
            max_iter=100000,
            tol=1e-3,
            random_state=42
        )

        model.fit(X_train, y_train)

        val_preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, val_preds))

        print("Fold RMSE:", rmse)

        rmse_scores.append(rmse)

        # accumulate averaged predictions
        test_predictions += model.predict(X_test) / kf.n_splits

    print("\nMean CV RMSE:", np.mean(rmse_scores))

    return test_predictions


def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp08_elasticnet_kfold_ensemble_20260323"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("Submission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = scale_features(X, X_test)

    preds = train_kfold_ensemble(X, y, X_test)

    print("\nSample predictions:", preds[:10])

    save_submission(test, preds)


if __name__ == "__main__":
    main()

Train shape: (1322, 1559)
Test shape: (550, 1558)

Training fold 1
Fold RMSE: 21.614823615792567

Training fold 2
Fold RMSE: 24.819446767938075

Training fold 3
Fold RMSE: 25.37001850551025

Training fold 4
Fold RMSE: 22.684033530418013

Training fold 5
Fold RMSE: 26.032619903209593

Mean CV RMSE: 24.104188464573703

Sample predictions: [189.77269699 178.82804117 170.40114343 163.8979588  154.79904887
 144.37719846 134.9711149  127.55508541 121.17755933 115.9138925 ]
Submission saved to: ../submissions/exp08_elasticnet_kfold_ensemble_20260323.csv
    0           1
0  95  189.772697
1  96  178.828041
2  97  170.401143
3  98  163.897959
4  99  154.799049
